## Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, lower, upper

## Read Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_customers")

## Silver Transformations
### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### City normalization

In [0]:
df = df.withColumn("customer_city", lower(col("customer_city")))

### State normalization

In [0]:
df = df.withColumn("customer_state", upper(col("customer_state")))

### Remove Records with Missing Customer ID

In [0]:
df = df.filter(col("customer_id").isNotNull())

### Remove duplicates

In [0]:
df = df.dropDuplicates(["customer_id"])

### Renaming Columns

In [0]:
RENAME_MAP = {
    "customer_zip_code_prefix": "zip_code_prefix",
    "customer_city": "city",
    "customer_state": "state_code",
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity checks of dataframe

In [0]:
df.limit(10).display()

## Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

### Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers LIMIT 10